In [ ]:
# Define the probabilistic context-free grammar (PCFG) with non-terminal symbols as keys
# and lists of production rules with associated probabilities as values.
pcfg = {
    "S": [('NP', 'VP', 1.0)],  # 'S' can be expanded to 'NP VP' with a probability of 1.0
    "PP": [('P', 'NP', 1.0)],  # 'PP' can be expanded to 'P NP' with a probability of 1.0
    "VP": [("V", "NP", 0.7), ("VP", "PP", 0.3)],  # 'VP' has two possible expansions with their probabilities
    "P": [("with", 1.0)],  # 'P' expands to the terminal 'with' with probability 1.0
    "V": [("saw", 1.0)],  # 'V' expands to the terminal 'saw' with probability 1.0
    "NP": [  # 'NP' can be expanded in different ways, each with associated probability
        ("NP", "PP", 0.4),
        ("astronomers", 0.1),
        ("ears", 0.1),
        ("saw", 0.04),
        ("stars", 0.18),
        ("telescopes", 0.1)
    ]
}

# Define the sentence to be parsed and its length
sentence = ["astronomers", "saw", "stars", "with", "ears"]
n = len(sentence)

# Initialize the table 'T' for dynamic programming.
# T is a 2D table (n x n), where each cell T[i][j] contains a dictionary to store
# possible non-terminals that can produce the substring sentence[i:j+1]
T = [[{} for _ in range(n)] for _ in range(n)]

# Fill diagonal elements of the table for individual words in the sentence.
# This step initializes the non-terminal probabilities for each single word in the sentence.
for i in range(n):
    for lhs, rules in pcfg.items():
        for rule in rules:
            if len(rule) == 2:  # Terminal production rule
                word, prob = rule
                if word == sentence[i]:  # If the terminal matches the sentence word at position i
                    T[i][i][lhs] = prob  # Store the probability for this non-terminal

# Fill the rest of the table using dynamic programming.
# We check spans of increasing length to find probabilities of non-terminals
# that can produce the substrings of the sentence.
for span in range(2, n + 1):  # span is the length of the substring
    for i in range(n - span + 1):  # starting index of the span
        j = i + span - 1  # ending index of the span
        for k in range(i, j):  # split the span into two parts at position k
            for lhs, rules in pcfg.items():
                for rule in rules:
                    if len(rule) == 3:  # Non-terminal production rule
                        A, B, prob = rule  # A and B are the components of the rule, with its probability
                        if A in T[i][k] and B in T[k + 1][j]:  # Check if T[i][k] and T[k+1][j] contain A and B
                            combined_prob = prob * T[i][k][A] * T[k + 1][j][B]  # Calculate combined probability
                            if lhs in T[i][j]:  # If lhs already has a probability, add to it
                                T[i][j][lhs] += combined_prob
                            else:  # Otherwise, initialize lhs with the combined probability
                                T[i][j][lhs] = combined_prob

# Print the table with probabilities for each possible non-terminal
# at each position in the sentence.
for row in T:
    print(row)
